In [7]:
:dep sha2 = "0.10"

In [12]:
use std::fs::File;
use std::io::{BufRead, BufReader};
use sha2::{Sha512, Digest};
use std::fs;

// --- 以下、ハッシュ計算ロジック ---

const CHARS: &[u8] = b"./0123456789ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz";

fn b64_from_24bit(b2: u8, b1: u8, b0: u8, n: usize, result: &mut String) {
    let mut w = (b2 as u32) << 16 | (b1 as u32) << 8 | (b0 as u32);
    for _ in 0..n {
        result.push(CHARS[(w & 0x3f) as usize] as char);
        w >>= 6;
    }
}

fn sha512crypt_encode(digest: &[u8]) -> String {
    let mut r = String::with_capacity(86);
    let b = digest;
    b64_from_24bit(b[0], b[21], b[42], 4, &mut r);
    b64_from_24bit(b[22], b[43], b[1], 4, &mut r);
    b64_from_24bit(b[44], b[2], b[23], 4, &mut r);
    b64_from_24bit(b[3], b[24], b[45], 4, &mut r);
    b64_from_24bit(b[25], b[46], b[4], 4, &mut r);
    b64_from_24bit(b[47], b[5], b[26], 4, &mut r);
    b64_from_24bit(b[6], b[27], b[48], 4, &mut r);
    b64_from_24bit(b[28], b[49], b[7], 4, &mut r);
    b64_from_24bit(b[50], b[8], b[29], 4, &mut r);
    b64_from_24bit(b[9], b[30], b[51], 4, &mut r);
    b64_from_24bit(b[31], b[52], b[10], 4, &mut r);
    b64_from_24bit(b[53], b[11], b[32], 4, &mut r);
    b64_from_24bit(b[12], b[33], b[54], 4, &mut r);
    b64_from_24bit(b[34], b[55], b[13], 4, &mut r);
    b64_from_24bit(b[56], b[14], b[35], 4, &mut r);
    b64_from_24bit(b[15], b[36], b[57], 4, &mut r);
    b64_from_24bit(b[37], b[58], b[16], 4, &mut r);
    b64_from_24bit(b[59], b[17], b[38], 4, &mut r);
    b64_from_24bit(b[18], b[39], b[60], 4, &mut r);
    b64_from_24bit(b[40], b[61], b[19], 4, &mut r);
    b64_from_24bit(b[62], b[20], b[41], 4, &mut r);
    b64_from_24bit(0, 0, b[63], 2, &mut r);
    r
}

pub fn sha512crypt(password: &[u8], salt: &[u8], rounds: u32) -> String {
    let plen = password.len();
    let slen = salt.len();

    let mut b = Sha512::new();
    b.update(password);
    b.update(salt);
    b.update(password);
    let digest_b = b.finalize();

    let mut a = Sha512::new();
    a.update(password);
    a.update(salt);
    let mut rem = plen;
    while rem > 64 { a.update(&digest_b[..]); rem -= 64; }
    a.update(&digest_b[..rem]);
    let mut bits = plen;
    while bits > 0 {
        if bits & 1 != 0 { a.update(&digest_b[..]); }
        else { a.update(password); }
        bits >>= 1;
    }
    let digest_a = a.finalize();

    let mut ph = Sha512::new();
    for _ in 0..plen { ph.update(password); }
    let dp = ph.finalize();
    let mut p_str = Vec::with_capacity(plen);
    let mut rem = plen;
    while rem > 64 { p_str.extend_from_slice(&dp); rem -= 64; }
    p_str.extend_from_slice(&dp[..rem]);

    let mut sh = Sha512::new();
    for _ in 0..(16usize + digest_a[0] as usize) { sh.update(salt); }
    let ds = sh.finalize();
    let mut s_str = Vec::with_capacity(slen);
    let mut rem = slen;
    while rem > 64 { s_str.extend_from_slice(&ds); rem -= 64; }
    s_str.extend_from_slice(&ds[..rem]);

    let mut c = digest_a.to_vec();
    for round in 0..rounds {
        let mut h = Sha512::new();
        if round & 1 != 0 { h.update(&p_str); } else { h.update(&c); }
        if round % 3 != 0 { h.update(&s_str); }
        if round % 7 != 0 { h.update(&p_str); }
        if round & 1 != 0 { h.update(&c); } else { h.update(&p_str); }
        c = h.finalize().to_vec();
    }

    sha512crypt_encode(&c)
}

fn parse_hash(s: &str) -> Option<(&str, &str)> {
    let parts: Vec<&str> = s.split('$').collect();
    if parts.len() < 4 || parts[1] != "6" { return None; }
    Some((parts[2], parts[3]))
}

// --- メイン処理 ---

fn main() {
    let shadow: &[(&str, &str)] = &[
        ("user00", "$6$Z4xEy/1KTCW.rz$Yxkc8XkscDusGWKan621H4eaPRjHc1bkXDjyFtcTtgxzlxvuPiE1rnqdQVO1lYgNOzg72FU95RQut93JF6Deo/"),
        ("user01", "$6$ffl1bXDBqKUiD$PoXP69PaxTTX.cgzYS6Tlj7UBvstr6JruGctoObFXCr4cYXjIbxBSMiQZiVkKvUxXUC23zP8PUyXjq6qEq63u1"),
        ("user02", "$6$ZsJXadT/rv$T/2gVzYwMBaAsZnHIjnUSmTozIF/ebMvtHIJjikFehvB8pvy28DUIQYbTJLG6QAxhzJAKOROnZq0xV4hUGefM1"),
        ("user03", "$6$l0NHH5FF0H/U$fPv3c5Cdls/UaZmglR4Qqh8vhpIBsmY1sEjHi486ZcDQ2Vx5GY0fcQYSorWj6l42jfI47w437n.NBm8NArFyT/"),
        ("user04", "$6$wAnAP/NMiLa/yE$.gi4r3xYuPTg5z2S59z2EzFbqpmwZYy1tBSVA9/hqTFnWY0tHqXbwL.dFQwHzKTuzXV6WMgjEZlyzUPGzVtPb0"),
        ("user05", "$6$jTgFhKHk/$xQIdn7snYAAGvifxC02YLXcAKkiuPbJ3KBkH2Q8BZ12TL2aepaUJotgfKfNSPCXWebyCY/skOmOymok.KIm5D0"),
        ("user06", "$6$8LXZt/zPbLtIn1o$ynsZxueG88Kz0vDr3cyK.21cv4GWw9iaW9oYZcmZ9SY5UpMQS1wl2/dbXGyR8WzVBKKP/6k8VYvWuiNQ3We52/"),
        ("user07", "$6$jnA8m/S5aU0/$PGrG8mDy.vs3W9xhG1qd56eOEainH9xntY48.duznt989TXMn6J.scOBqp4BWg3fHWxoFgBn26LYvcnqWGcoF1"),
        ("user08", "$6$ITB7n/qsP$fmrmItHX9B96PmhsxIX21vdYDvFHiIPnyzRFjWIbcd3y/DRHCm0lzyJEnWlQChdDAiFUFXtqwoTbEdREXQ99M."),
        ("user09", "$6$LpgLJrjPV$6sa0KW08Q10S.C/BSUHlHaQZT5n8uIygZSsWP5drdmuhI7c17wWCK/GEzQS7g8EL//5bqdjo1C90smTDhLEcF1"),
        ("user10", "$6$0VSPwOzcL//6QR$RgtMpkfVPb5Cli7cjVE5jMgJlN10xY1R3jxRNrY0l/84R3.NvxP3I8XtkMkonU6DKhge0JGp54DZLQqUN9kL7/"),
        ("user11", "$6$zryub/lvSKj7Xl$eazV2fmcJa5M3qMovQqARGK59Qxtfv2zjUJvphKNnyUMVyBn.SjEFhRT/mAjz3QFroNbwmrYLtrpyxjH.q64n/"),
        ("user12", "$6$tAkM0dDUFe76d8K/$OnNGFEuIf1seMlLHb.8.y5/cpmBUcMbhLhOfFdd0E/DKASXPS4riB4uz2Fg3om9Atg.g7s.JFoKV0uuJ461KV/"),
        ("user13", "$6$0cCdE5Nfqu/HFS$PwnLdS.chtm6qGwf2Uuiko7V3fMwjcQ52M8hslvoReFQ9XOBXw603Ok20VJwWAwR6RNv6adn6a6kuRm5Y3.ge1"),
        ("user14", "$6$RgPs7j4eSa/v$71CeLB9Z1Fafi6vi2ou5LzRz5xXWTzvZeZgelnm2przx.JQYp21p8h2BCyTYFd10MKD/cquPvn42vSzlJJJ8Q1"),
        ("user15", "$6$1uhGQ/5DwMp/$UjYTEVaChEzmUITvWpaZVvYYDLBULpI4IEyieClSsyC2NHwEnaDx6xwtUVpQPxEhi6R7OQhX68Oo5CfilYqDQ."),
        ("user16", "$6$V/InSacMp8U$UpDgdL/GS/kdFmn1rO97YkLAeTgofu4fDVUGoV1PWnVFxUtVyx24ix5hJp53FkBuqdzmXgwGcb6MU5AWJWjaB1"),
        ("user17", "$6$d6mWSrE8vxDe$UqTgKPfKxm0/Aboz8DeFNNiZsFBYyE6iGpqUzSX4UpWSDfXt1DERBtI29H2Gz5q.6ls3730naAo31wAacvs/L0"),
        ("user18", "$6$ulcKu/ddomcNGRJj$i8XB1D4YtLGbAHX0XHX88ObUWw8dQsrTqoliGAU//zGHNLmLeWd.4k5YHViNSy3rlGTQSRPtutlKnub8aRnzy0"),
        ("user19", "$6$cVnhE9CwfSIIA$wrn6p3cgfz.JOc6KVkieNCtc.FzkjUdcDDlivn0APnYv9/z4tt7hUpPft5T8kMmnx/hiF92vjnDxcauVyQySp."),
        ("user20", "$6$2Pg2VxXg$K8AqsCMPAFiXSxNjETBWqEHQom9Q5dDIz9/nItxpQatrG9gvv9CRJP3kQzKLbRf13FxfOXpeEYIpOEK.2i1HP0"),
    ];

    println!("Loading dictionary file...");
    let dict_file = File::open("dicti0nary_8Th64ikELWEsZFrf.txt").expect("辞書ファイルが見つかりません");
    let reader = BufReader::new(dict_file);
    let dictionary: Vec<String> = reader.lines().filter_map(|l| l.ok()).collect();
    println!("Loaded {} words.\n", dictionary.len());

    let mut results: Vec<(u32, char)> = Vec::new();

    for (user, hash_field) in shadow {
        let num: u32 = user[4..].parse().unwrap();
        if let Some((salt, expected)) = parse_hash(hash_field) {
            for word in &dictionary {
                if sha512crypt(word.as_bytes(), salt.as_bytes(), 5000) == expected {
                    let ch = word.chars().next().unwrap_or(' ');
                    println!("{}: {}", user, word);
                    results.push((num, ch));
                    break;
                }
            }
        }
    }

    results.sort_by_key(|&(n, _)| n);
    let flag: String = results.iter().map(|&(_, c)| c).collect();
    fs::write("flag.txt", flag).unwrap();
}

main();

Loading dictionary file...
Loaded 3042 words.

user00: FREQUENT
user01: LATTER
user02: ADDITIONAL
user03: GENDER
user04: __________
user05: applies
user06: SPIRITS
user07: independent
user08: ultimate
user09: JENNY
user10: HELD
user11: SUFFERS
user12: LEAVE
user13: floating
user14: zecht
user15: opinion
user16: QUESTION
user17: karaoke
user18: strange
user19: zero
user20: DELIGHT
